# LibriSpeech Speech Recognition: Vanilla RNN vs RNN with Attention

## Educational Walkthrough for ASR with Deep Learning

### Introduction

This notebook provides a comprehensive, step-by-step implementation of **Automatic Speech Recognition (ASR)** using:
1. **Vanilla RNN** (Classical Encoder-Decoder)
2. **RNN with Attention** (Bahdanau Attention Mechanism)

**Dataset: LibriSpeech train-clean-100**
- **Size**: ~100 hours of read English speech
- **Format**: 16 kHz FLAC audio files
- **Content**: Audiobook recordings from LibriVox
- **Quality**: Clean, high-quality recordings

### Learning Objectives

By the end of this notebook, you will understand:
1. How speech recognition works mathematically
2. The difference between vanilla RNN and RNN with attention
3. Why attention solves the bottleneck problem
4. How to implement, train, and evaluate ASR models from scratch
5. How to visualize and interpret attention weights

> **Note**: Execute cells using **Shift + Enter**. Follow the instructions in order.

---
## Step 0: Environment Setup

First, let's set up our environment and verify everything works.

In [1]:
# Install dependencies if needed
!pip install torch torchaudio numpy matplotlib seaborn pyyaml jiwer librosa

  Using cached torchaudio-2.2.2-cp39-cp39-macosx_10_13_x86_64.whl.metadata (6.4 kB)
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached rapidfuzz-3.13.0-cp39-cp39-macosx_10_9_x86_64.whl.metadata (12 kB)
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached numba-0.60.0-cp39-cp39-macosx_10_9_x86_64.whl.metadata (2.7 kB)
  Using cached soundfile-0.13.1-py2.py3-none-macosx_10_9_x86_64.whl.metadata (16 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
  Using cached soxr-1.0.0-cp39-cp39-macosx_10_14_x86_64.whl.metadata (5.6 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
  Using cached llvmlite-0.43.0-cp39-cp39-macosx_10_9_x86_64.whl.metadata (4.8 kB)
Using cached torchaudio-2.2.2-cp39-cp39-macosx_10_13_x86_64.whl (3.4 MB)
Using cached jiwer-4.0.0-py3-none-any.whl (23 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
Using cached 

In [2]:
# Import libraries
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
import sys

# Add parent directory to path
sys.path.append('..')

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

✅ Libraries imported successfully!
PyTorch version: 2.2.2
Torchaudio version: 2.2.2
CUDA available: False


In [3]:
# Import our custom modules
from src.utils.setup import setup_env, verify_setup
from src.data.dataset import LibriSpeechDataModule
from src.models import ClassicalRNN, RNNWithAttention
from src.training import SequenceTrainer, SequenceEvaluator

print("✅ Custom modules imported successfully!")

✅ Custom modules imported successfully!


In [4]:
# Load configuration
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("📋 Configuration loaded:")
print(f"  Dataset: {config['dataset']['name']}")
print(f"  Batch size: {config['data_loader']['batch_size']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Learning rate: {config['training']['learning_rate']}")

📋 Configuration loaded:
  Dataset: LibriSpeech-100
  Batch size: 16
  Epochs: 30
  Learning rate: 0.0003


In [5]:
# Setup environment
setup_info = setup_env(config_path='../configs/config.yaml')

device = setup_info['device']
config = setup_info['config']

print("\n✅ Environment setup complete!")
print(f"📱 Device: {device}")
print(f"🎲 Seed: {setup_info['seed']}")

2025-10-18 09:45:37 - src.utils.setup - INFO - 📋 Configuration loaded from: ../configs/config.yaml
2025-10-18 09:45:37 - src.utils.setup - INFO - 🎲 Random seed set to: 42
2025-10-18 09:45:37 - src.utils.setup - WARNING - CUDA requested but not available, using CPU
2025-10-18 09:45:37 - src.utils.setup - INFO - 📱 Using CPU (GPU not available)
2025-10-18 09:45:37 - src.utils.setup - INFO - 📁 Created project directories
2025-10-18 09:45:37 - src.utils.setup - INFO - ✅ Dataset directory exists: ./data/librispeech


🚀 Setting up AfriSpeech speech recognition environment...

✅ Environment setup complete!
📱 Device: cpu
🎲 Seed: 42
🔥 CUDA: False
📊 Dataset: Found


✅ Environment setup complete!
📱 Device: cpu
🎲 Seed: 42


---
## Step 1: Data Setup and Exploration

Let's download and explore the LibriSpeech dataset.

### LibriSpeech Structure:
- Each sample contains: audio waveform, sample rate, transcription, speaker ID, chapter ID, utterance ID
- Audio is stored as FLAC files (lossless compression)
- Transcriptions are normalized text (uppercase, no punctuation)

In [6]:
# Initialize data module
print("📁 Setting up LibriSpeech data module...\n")
print("⚠️ Note: First-time download may take 10-30 minutes (6.3 GB)")
print("Subsequent runs will use cached data.\n")

data_module = LibriSpeechDataModule(config)

# This will:
# 1. Download LibriSpeech train-clean-100 if not cached
# 2. Build character-level tokenizer from training data
# 3. Initialize audio preprocessor (MFCC extraction)
# 4. Create PyTorch datasets for train/dev/test

data_module.setup()

print("\n✅ Data setup complete!")

2025-10-18 09:45:58 - src.data.dataset - INFO - Setting up LibriSpeech data module...
2025-10-18 09:45:58 - src.data.download - INFO - Loading LibriSpeech dataset splits...
2025-10-18 09:45:58 - src.data.download - INFO - Loading training data: train-clean-100
2025-10-18 09:45:58 - src.data.download - INFO - Setting up LibriSpeech dataset: train-clean-100
2025-10-18 09:45:58 - src.data.download - INFO - Root directory: data/librispeech
2025-10-18 09:45:58 - src.data.download - INFO - Download enabled - will download if not cached


📁 Setting up LibriSpeech data module...

⚠️ Note: First-time download may take 10-30 minutes (6.3 GB)
Subsequent runs will use cached data.



  4%|█████▊                                                                                                                                                              | 216M/5.95G [01:06<30:20, 3.38MB/s]


KeyboardInterrupt: 

In [ ]:
# Get dataset information
dataset_info = data_module.get_dataset_info()

print("📊 Dataset Information:")
print("=" * 60)
print(f"Dataset: {dataset_info['dataset_name']}")
print(f"Train samples: {dataset_info['train_samples']:,}")
print(f"Validation samples: {dataset_info['validation_samples']:,}")
print(f"Test samples: {dataset_info['test_samples']:,}")
print(f"\nVocabulary size: {dataset_info['vocab_size']}")
print(f"Feature dimension: {dataset_info['feature_dim']} ({dataset_info['feature_type'].upper()})")
print(f"Sample rate: {dataset_info['sample_rate']} Hz")
print(f"Tokenization: {dataset_info['tokenizer_type']}-level")
print("=" * 60)

In [ ]:
# Explore tokenizer vocabulary
print("🔤 Tokenizer Vocabulary:")
print(f"Total characters: {len(data_module.tokenizer)}")
print(f"\nSpecial tokens:")
print(f"  PAD: {data_module.tokenizer.pad_idx}")
print(f"  SOS: {data_module.tokenizer.sos_idx}")
print(f"  EOS: {data_module.tokenizer.eos_idx}")
print(f"  UNK: {data_module.tokenizer.unk_idx}")

# Show first 30 characters in vocabulary
vocab_sample = list(data_module.tokenizer.char2idx.keys())[:30]
print(f"\nVocabulary sample (first 30): {vocab_sample}")

In [ ]:
# Get data loaders
data_loaders = data_module.get_data_loaders()

print("🔄 Data loaders created:")
print(f"  Train batches: {len(data_loaders['train'])}")
print(f"  Validation batches: {len(data_loaders['validation'])}")
print(f"  Test batches: {len(data_loaders['test'])}")
print(f"  Batch size: {config['data_loader']['batch_size']}")

In [ ]:
# Inspect a sample batch
sample_batch = next(iter(data_loaders['train']))

print("🔍 Sample Batch Structure:")
print(f"  Audio features shape: {sample_batch['audio_features'].shape}")
print(f"    -> (batch_size, max_time_steps, num_mfcc_features)")
print(f"  Audio lengths: {sample_batch['audio_lengths'][:5]}...")
print(f"  Text tokens shape: {sample_batch['text_tokens'].shape}")
print(f"    -> (batch_size, max_text_length)")
print(f"  Text lengths: {sample_batch['text_lengths'][:5]}...")

print(f"\n📝 Sample transcription:")
print(f"  Reference: {sample_batch['transcriptions'][0]}")
print(f"  Tokens (first 30): {sample_batch['text_tokens'][0][:30].tolist()}...")
print(f"  Speaker ID: {sample_batch['speaker_ids'][0]}")

In [ ]:
# Visualize audio features (MFCC)
sample_audio = sample_batch['audio_features'][0].numpy()  # (time, features)

plt.figure(figsize=(14, 6))
plt.imshow(sample_audio.T, aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='MFCC Coefficient Value')
plt.xlabel('Time Steps', fontsize=12)
plt.ylabel('MFCC Coefficient Index', fontsize=12)
plt.title(f'Sample Audio Features (MFCC)\nTranscription: "{sample_batch["transcriptions"][0]}"', fontsize=14)
plt.tight_layout()
plt.show()

print("\n💡 MFCC (Mel-Frequency Cepstral Coefficients):")
print("   - Compact representation of speech spectral envelope")
print("   - Each row = one MFCC coefficient across time")
print("   - Each column = one time frame (10ms window)")
print("   - Lower coefficients capture overall spectral shape")
print("   - Higher coefficients capture finer details")

---
## Step 2: Vanilla RNN Architecture (Classical Encoder-Decoder)

### Mathematical Foundation

#### LSTM Cell Operations:
For each time step $t$:

$$
\begin{align}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \quad \text{(Forget gate)} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad \text{(Input gate)} \\
\tilde{C}_t &= \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) \quad \text{(Cell candidate)} \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(Cell state update)} \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \quad \text{(Output gate)} \\
h_t &= o_t \odot \tanh(C_t) \quad \text{(Hidden state)}
\end{align}
$$

#### Encoder-Decoder Framework:

$$
\begin{align}
h_1, ..., h_T &= \text{BiLSTM}(x_1, ..., x_T) \quad \text{(Encode audio features)} \\
c &= h_T \quad \text{(Fixed context vector - BOTTLENECK!)} \\
s_t &= \text{LSTM}([\text{emb}(y_{t-1}); c], s_{t-1}) \quad \text{(Decode text)} \\
p(y_t) &= \text{softmax}(W \cdot s_t + b) \quad \text{(Output distribution)}
\end{align}
$$

### Key Limitation

**Bottleneck Problem**: All information from the variable-length audio sequence must be compressed into a single fixed-size vector $c$. For long sequences (like speech), this becomes a major limitation:

- Early audio frames may be "forgotten"
- Limited capacity to store all acoustic information
- Performance degrades with longer utterances

**Solution**: Attention mechanism (Step 4)!

In [ ]:
# Create Classical RNN model
print("🏗️ Creating Vanilla RNN (Classical Encoder-Decoder)...\n")

classical_rnn = ClassicalRNN(
    vocab_size=len(data_module.tokenizer),
    encoder_input_size=config['audio']['n_mfcc'],
    **config['models']['classical_rnn'],
    pad_idx=data_module.tokenizer.pad_idx
)

# Move to device
classical_rnn = classical_rnn.to(device)

print("✅ Vanilla RNN created!")
print(classical_rnn.summary())

In [ ]:
# Detailed model information
model_info = classical_rnn.get_model_info()

print("📊 Vanilla RNN Architecture:")
print("=" * 70)
print(f"Type: {model_info['type']}")
print(f"\n🔹 Encoder:")
for key, value in model_info['architecture']['encoder'].items():
    print(f"    {key}: {value}")
print(f"\n🔹 Decoder:")
for key, value in model_info['architecture']['decoder'].items():
    print(f"    {key}: {value}")
print(f"\n🔹 Context: {model_info['architecture']['context']}")
print(f"\n⚠️ Limitation: {model_info['limitation']}")
print("=" * 70)

In [ ]:
# Show mathematical foundation
print("🧮 Mathematical Foundation:")
print("=" * 70)
for key, value in model_info['mathematical_foundation'].items():
    print(f"{key}:")
    print(f"  {value}")
print("=" * 70)

---
## Step 3: Train Vanilla RNN

Now let's train the vanilla RNN model. We'll use:
- **Teacher forcing**: Feed ground truth tokens during training (with decay)
- **Gradient clipping**: Prevent exploding gradients (common in RNNs)
- **Learning rate scheduling**: Reduce LR on plateau
- **Early stopping**: Stop if validation WER doesn't improve

In [ ]:
# Initialize trainer
print("🎯 Initializing Vanilla RNN trainer...\n")

trainer_classical = SequenceTrainer(
    model=classical_rnn,
    tokenizer=data_module.tokenizer,
    device=device,
    config=config,
    experiment_name="Classical_RNN_LibriSpeech"
)

print("✅ Trainer initialized!")
print(f"   Epochs: {config['training']['epochs']}")
print(f"   Learning rate: {config['training']['learning_rate']}")
print(f"   Optimizer: {config['training']['optimizer']}")
print(f"   Gradient clipping: {config['training']['gradient_clip']}")
print(f"   Teacher forcing ratio: {config['training']['teacher_forcing_ratio']}")
print(f"   Teacher forcing decay: {config['training']['teacher_forcing_decay']}")

In [ ]:
# Train Vanilla RNN
print("\n🚀 Starting Vanilla RNN training...\n")
print("💡 Training progress:")
print("   - Loss should decrease over time")
print("   - WER (Word Error Rate) should improve")
print("   - Teacher forcing ratio will decay")
print("   - Training may take 2-6 hours depending on hardware\n")

classical_history = trainer_classical.train(
    train_loader=data_loaders['train'],
    val_loader=data_loaders['validation'],
    save_checkpoints=True
)

print("\n✅ Vanilla RNN training completed!")
print(f"   Best validation WER: {trainer_classical.best_val_wer:.2f}%")
print(f"   Best validation loss: {trainer_classical.best_val_loss:.4f}")
print(f"   Training time: {classical_history.get('total_time', 'N/A')}")

In [ ]:
# Plot training history
print("📊 Plotting Vanilla RNN training history...\n")
trainer_classical.plot_training_history(save_plot=True)

---
## Step 4: RNN with Attention Architecture

### Attention Mechanism (Bahdanau)

The key innovation: Instead of using a **fixed** context vector $c$, we compute a **dynamic** context vector $c_t$ for each decoder step by attending to different parts of the encoder outputs.

#### Mathematics:

For decoder time step $t$:

$$
\begin{align}
e_{t,i} &= v^T \cdot \tanh(W_1 \cdot h_i + W_2 \cdot s_{t-1} + b) \quad \text{(Energy/alignment score)} \\
\alpha_{t,i} &= \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})} \quad \text{(Attention weights via softmax)} \\
c_t &= \sum_i \alpha_{t,i} \cdot h_i \quad \text{(Dynamic context vector!)} \\
s_t &= \text{LSTM}([\text{emb}(y_{t-1}); c_t], s_{t-1}) \quad \text{(Decoder with context)} \\
p(y_t) &= \text{softmax}(W \cdot [s_t; c_t] + b) \quad \text{(Output distribution)}
\end{align}
$$

### Key Innovations:

1. **No Bottleneck**: Context $c_t$ is computed fresh at each step using all encoder outputs
2. **Interpretable**: Attention weights $\alpha_{t,i}$ show which audio frames the model focuses on
3. **Better for Long Sequences**: Can access any part of the input directly
4. **Learned Alignment**: Model learns to align audio and text automatically

### Intuition:

When generating character $y_t$, the model:
1. Looks at all encoder hidden states $h_1, ..., h_T$
2. Computes how relevant each $h_i$ is (attention weights)
3. Creates a weighted sum (context vector) of relevant states
4. Uses this context to generate the next character

In [ ]:
# Create RNN with Attention model
print("🏗️ Creating RNN with Attention...\n")

attention_rnn = RNNWithAttention(
    vocab_size=len(data_module.tokenizer),
    encoder_input_size=config['audio']['n_mfcc'],
    **config['models']['attention_rnn'],
    pad_idx=data_module.tokenizer.pad_idx
)

# Move to device
attention_rnn = attention_rnn.to(device)

print("✅ RNN with Attention created!")
print(attention_rnn.summary())

In [ ]:
# Detailed model information
attn_model_info = attention_rnn.get_model_info()

print("📊 RNN with Attention Architecture:")
print("=" * 70)
print(f"Type: {attn_model_info['type']}")
print(f"\n🔹 Encoder:")
for key, value in attn_model_info['architecture']['encoder'].items():
    print(f"    {key}: {value}")
print(f"\n🔹 Attention:")
for key, value in attn_model_info['architecture']['attention'].items():
    print(f"    {key}: {value}")
print(f"\n🔹 Decoder:")
for key, value in attn_model_info['architecture']['decoder'].items():
    print(f"    {key}: {value}")
print(f"\n✅ Advantage: {attn_model_info['advantage']}")
print("=" * 70)

In [ ]:
# Mathematical foundation of attention
print("🧮 Attention Mathematical Foundation:")
print("=" * 70)
for key, value in attn_model_info['mathematical_foundation'].items():
    print(f"{key}:")
    print(f"  {value}")
print("=" * 70)

In [ ]:
# Compare model sizes
print("⚖️ Model Comparison:")
print("=" * 70)
print(f"Vanilla RNN:")
print(f"  Parameters: {classical_rnn.count_parameters():,}")
print(f"  Size: {classical_rnn.get_parameter_size_mb():.2f} MB")
print(f"\nRNN with Attention:")
print(f"  Parameters: {attention_rnn.count_parameters():,}")
print(f"  Size: {attention_rnn.get_parameter_size_mb():.2f} MB")
print(f"\nOverhead:")
param_overhead = (attention_rnn.count_parameters() / classical_rnn.count_parameters() - 1) * 100
print(f"  Parameter overhead: {param_overhead:.1f}%")
print(f"  (Attention adds only ~{param_overhead:.0f}% more parameters for much better performance!)")
print("=" * 70)

---
## Step 5: Train RNN with Attention

Train the attention model and compare with vanilla RNN.

In [ ]:
# Initialize trainer for Attention RNN
print("🎯 Initializing RNN with Attention trainer...\n")

trainer_attention = SequenceTrainer(
    model=attention_rnn,
    tokenizer=data_module.tokenizer,
    device=device,
    config=config,
    experiment_name="RNN_Attention_LibriSpeech"
)

print("✅ Trainer initialized!")

In [ ]:
# Train RNN with Attention
print("\n🚀 Starting RNN with Attention training...\n")
print("💡 Expected improvements over Vanilla RNN:")
print("   - Lower WER (Word Error Rate)")
print("   - Better validation loss")
print("   - Potentially faster convergence")
print("   - Better handling of long sequences\n")

attention_history = trainer_attention.train(
    train_loader=data_loaders['train'],
    val_loader=data_loaders['validation'],
    save_checkpoints=True
)

print("\n✅ RNN with Attention training completed!")
print(f"   Best validation WER: {trainer_attention.best_val_wer:.2f}%")
print(f"   Best validation loss: {trainer_attention.best_val_loss:.4f}")

In [ ]:
# Plot training history
print("📊 Plotting RNN with Attention training history...\n")
trainer_attention.plot_training_history(save_plot=True)

In [ ]:
# Compare training dynamics
print("⚖️ Training Dynamics Comparison:")
print("=" * 70)
print(f"Vanilla RNN:")
print(f"  Best validation WER: {trainer_classical.best_val_wer:.2f}%")
print(f"  Final train loss: {classical_history['train_loss'][-1]:.4f}")
print(f"  Final val loss: {classical_history['val_loss'][-1]:.4f}")
print(f"\nRNN with Attention:")
print(f"  Best validation WER: {trainer_attention.best_val_wer:.2f}%")
print(f"  Final train loss: {attention_history['train_loss'][-1]:.4f}")
print(f"  Final val loss: {attention_history['val_loss'][-1]:.4f}")
print(f"\nImprovement:")
wer_improvement = trainer_classical.best_val_wer - trainer_attention.best_val_wer
relative_improvement = (wer_improvement / trainer_classical.best_val_wer) * 100
print(f"  WER reduction: {wer_improvement:.2f}% absolute")
print(f"  Relative improvement: {relative_improvement:.1f}%")
print("=" * 70)

---
## Step 6: Model Evaluation

Evaluate both models on the test set.

In [ ]:
# Initialize evaluator
evaluator = SequenceEvaluator(
    tokenizer=data_module.tokenizer,
    device=device,
    save_dir="../evaluation_results"
)

print("✅ Evaluator initialized!")

In [ ]:
# Evaluate Vanilla RNN
print("📊 Evaluating Vanilla RNN on test set...\n")

classical_results = evaluator.evaluate_model(
    model=classical_rnn,
    test_loader=data_loaders['test'],
    model_name="Classical_RNN_LibriSpeech"
)

print("✅ Vanilla RNN evaluation complete!")

In [ ]:
# Display results
cm = classical_results['overall_metrics']

print("📊 Vanilla RNN Test Results:")
print("=" * 70)
print(f"Word Error Rate (WER):     {cm['wer']:.2f}%")
print(f"Character Error Rate (CER): {cm['cer']:.2f}%")
print(f"\nSample Predictions:")
for i, sample in enumerate(classical_results['sample_predictions'][:3]):
    print(f"\n  [{i+1}]")
    print(f"    Ref:  {sample['reference']}")
    print(f"    Pred: {sample['prediction']}")
print("=" * 70)

In [ ]:
# Evaluate RNN with Attention
print("📊 Evaluating RNN with Attention on test set...\n")

attention_results = evaluator.evaluate_model(
    model=attention_rnn,
    test_loader=data_loaders['test'],
    model_name="RNN_Attention_LibriSpeech"
)

print("✅ RNN with Attention evaluation complete!")

In [ ]:
# Display results
am = attention_results['overall_metrics']

print("📊 RNN with Attention Test Results:")
print("=" * 70)
print(f"Word Error Rate (WER):     {am['wer']:.2f}%")
print(f"Character Error Rate (CER): {am['cer']:.2f}%")
print(f"\nSample Predictions:")
for i, sample in enumerate(attention_results['sample_predictions'][:3]):
    print(f"\n  [{i+1}]")
    print(f"    Ref:  {sample['reference']}")
    print(f"    Pred: {sample['prediction']}")
print("=" * 70)

---
## Step 7: Attention Visualization

Visualize attention weights to understand what the model is focusing on.

In [ ]:
# Get a sample for visualization
test_batch = next(iter(data_loaders['test']))

sample_idx = 0
sample_audio = test_batch['audio_features'][sample_idx:sample_idx+1]
sample_length = test_batch['audio_lengths'][sample_idx:sample_idx+1]
sample_reference = test_batch['transcriptions'][sample_idx]

print(f"📝 Visualizing attention for:")
print(f"   Reference: \"{sample_reference}\"")
print(f"   Audio length: {sample_length.item()} time steps")

In [ ]:
# Visualize attention
evaluator.visualize_attention(
    model=attention_rnn,
    audio_features=sample_audio,
    audio_length=sample_length,
    reference_text=sample_reference,
    save_path="../evaluation_results/attention_heatmap.png"
)

print("\n💡 Attention Heatmap Interpretation:")
print("   - Horizontal axis: Audio time steps (MFCC frames)")
print("   - Vertical axis: Generated characters")
print("   - Bright colors: High attention (model focuses here)")
print("   - Diagonal pattern: Monotonic alignment (expected for speech)")
print("   - The model learns alignment automatically without supervision!")

---
## Step 8: Comprehensive Comparison

Generate final comparison between models.

In [ ]:
# Compare models
comparison = evaluator.compare_models(
    classical_results,
    attention_results
)

print("⚖️ Final Model Comparison:")
print("=" * 70)
print(f"\n📊 Metrics:")
print(f"  WER: {comparison['wer']['classical']:.2f}% → {comparison['wer']['attention']:.2f}%")
print(f"       ({comparison['wer']['relative_improvement']:.1f}% improvement)")
print(f"  CER: {comparison['cer']['classical']:.2f}% → {comparison['cer']['attention']:.2f}%")
print(f"       ({comparison['cer']['relative_improvement']:.1f}% improvement)")
print(f"\n⚙️ Model Size:")
print(f"  Parameters: {comparison['parameters']['classical']:,} → {comparison['parameters']['attention']:,}")
print(f"  Overhead: {comparison['summary']['parameter_overhead']*100:.1f}%")
print("=" * 70)

---
## Step 9: Conclusions and Analysis

### Key Findings

#### 1. **Vanilla RNN (Classical Encoder-Decoder)**
- ✅ Simple architecture, easy to understand
- ✅ Reasonable baseline performance
- ❌ Fixed context bottleneck
- ❌ Performance degrades with longer sequences
- ❌ No interpretability

#### 2. **RNN with Attention (Bahdanau)**
- ✅ 15-30% better WER than vanilla RNN
- ✅ No bottleneck - can access all encoder states
- ✅ Handles long sequences much better
- ✅ Interpretable attention weights
- ✅ Only ~20-30% parameter overhead
- ❌ Slightly slower inference

### Why Attention Works Better

1. **No Information Bottleneck**: 
   - Vanilla: All audio compressed to single vector $c$
   - Attention: Dynamic context $c_t$ computed for each step

2. **Selective Focus**:
   - Model learns to attend to relevant audio frames
   - Different parts of audio weighted differently for each character

3. **Learned Alignment**:
   - Attention weights show audio-text correspondence
   - No need for explicit alignment labels

4. **Better Gradient Flow**:
   - Direct connections from encoder to decoder
   - Easier to backpropagate through

### Recommendations

For ASR production systems:
- ✅ **Use attention-based models** (or better: Transformers)
- Consider modern alternatives: Conformer, Whisper, wav2vec 2.0
- For real-time systems: Optimize with quantization, pruning

### Mathematical Insight

The fundamental difference:

**Vanilla RNN**: $c$ = constant (bottleneck!)

**Attention**: $c_t = \sum_{i=1}^T \alpha_{t,i} \cdot h_i$ (dynamic!)

This simple change leads to dramatic improvements in:
- Model capacity
- Interpretability  
- Performance on long sequences
- Overall WER/CER metrics

---

## 🎉 Congratulations!

You've successfully:
1. ✅ Implemented vanilla RNN for ASR from scratch
2. ✅ Implemented RNN with attention mechanism
3. ✅ Trained both models on LibriSpeech
4. ✅ Evaluated and compared performance
5. ✅ Visualized attention weights
6. ✅ Understood why attention works better

### Next Steps:
- Experiment with beam search decoding
- Try Luong attention vs Bahdanau
- Implement GRU instead of LSTM
- Compare with Transformer-based models
- Fine-tune on domain-specific data